# Shinhan Card 파일 불러오기

## 0. 초기 세팅
각종 라이브러리를 불러오고 초기값 등을 세팅합니다.

In [1]:
#!uv add xlrd
#!uv add lxml

In [2]:
from pathlib import Path
import pandas as pd

In [3]:
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"
SHINHAN_CARD_XLS = DATA_DIR / "shinhancard.xls"

## 1. 데이터 파일 조회 및 전처리

In [38]:
sh_df = pd.read_excel(SHINHAN_CARD_XLS)
len(sh_df)

8

In [39]:
sh_df

,거래일,카드구분,이용카드,가맹점명,승인번호,금액,매입구분,이용구분,거래통화,해외이용금액,취소상태
0,2026.01.24 11:08,신용,본인204*,홈플러스㈜,25863165.0,68210,결제확정,일시불,NaN,NaN,NaN
1,2026.01.16 21:55,신용,본인204*,홈플러스㈜,31299218.0,50240,결제확정,일시불,NaN,NaN,NaN
2,2026.01.16 12:35,신용,본인204*,(주)아트박스 강서 홈플러스점,23635058.0,2000,결제확정,일시불,NaN,NaN,NaN
3,2026.01.07 16:48,신용,본인204*,새서울약국,8159190.0,5700,결제확정,일시불,NaN,NaN,NaN
4,2026.01.07 16:44,신용,본인204*,코아이비인후과,8112630.0,10800,결제확정,일시불,NaN,NaN,NaN
5,2026.01.07 12:16,신용,본인204*,강서홈약국,4578218.0,6000,결제확정,일시불,NaN,NaN,NaN
6,2026.01.03 08:36,신용,본인204*,홈플러스㈜,4037845.0,70940,결제확정,일시불,NaN,NaN,NaN
7,NaN,NaN,NaN,총 7건,NaN,213890,NaN,NaN,NaN,NaN,NaN


In [40]:
sh_df["승인번호"]= (sh_df["승인번호"].dropna()
.astype("int64")
.astype("str"))

In [41]:
sh_df["거래일"] = pd.to_datetime(sh_df["거래일"], errors="coerce").dt.date


In [ ]:
text_cols = [
    "가맹점명",
    "이용구분",
    "매입구분"

]

for col in text_cols:
    sh_df[col] = sh_df[col].astype("string")

In [43]:
sh_df["금액"] = pd.to_numeric(sh_df["금액"], errors="coerce").fillna(0).astype("int64")

In [46]:
sh_df = sh_df.iloc[:-1]
sh_df

,거래일,카드구분,이용카드,가맹점명,승인번호,금액,매입구분,이용구분,거래통화,해외이용금액,취소상태
0,2026-01-24,신용,본인204*,홈플러스㈜,25863165,68210,결제확정,일시불,NaN,NaN,NaN
1,2026-01-16,신용,본인204*,홈플러스㈜,31299218,50240,결제확정,일시불,NaN,NaN,NaN
2,2026-01-16,신용,본인204*,(주)아트박스 강서 홈플러스점,23635058,2000,결제확정,일시불,NaN,NaN,NaN
3,2026-01-07,신용,본인204*,새서울약국,8159190,5700,결제확정,일시불,NaN,NaN,NaN
4,2026-01-07,신용,본인204*,코아이비인후과,8112630,10800,결제확정,일시불,NaN,NaN,NaN
5,2026-01-07,신용,본인204*,강서홈약국,4578218,6000,결제확정,일시불,NaN,NaN,NaN
6,2026-01-03,신용,본인204*,홈플러스㈜,4037845,70940,결제확정,일시불,NaN,NaN,NaN


## 2. DB 데이터로 매핑

In [ ]:
# 신한카드 엑셀 데이터프레임을 가계부 지출 데이터프레임으로 매핑하는 함수
# sql/ddl/expenditure.sql 의 컬럼과 매핑됨

EXCHANGE_RATE_USD_TO_KRW = 1450
PAYMENT_TYPE = "card"
CARD_COMPANY = "shinhan"

def map_shinhan_card_df_to_expenditure(df: pd.DataFrame) -> pd.DataFrame:
    """신한카드 엑셀 데이터프레임을 가계부 지출 데이터프레임으로 매핑합니다."""
    mapped_df = pd.DataFrame()
    mapped_df["used_at"] = df["거래일"]
    mapped_df["payment_type"] = PAYMENT_TYPE
    mapped_df["payment_provider"] = CARD_COMPANY
    mapped_df["merchant_name"] = df["가맹점명"]

    mapped_df["installment_type"] = df["이용구분"].map(
        lambda x: "single" if x == "일시불" else "installment"
    )

    mapped_df["amount"] = df["금액"]
    mapped_df["remaining_amount"] = 0
    mapped_df["category"] = "unknown"
    mapped_df["memo"] = None
    mapped_df["source_uid"] = (
        PAYMENT_TYPE + "_" + CARD_COMPANY + "_" + df["승인번호"]
    )

    return mapped_df


In [48]:
expenditure_df = map_shinhan_card_df_to_expenditure(sh_df)
expenditure_df.head()

,used_at,payment_type,card_company,merchant_name,installment_type,amount,remaining_amount,category,memo,source_uid
0,2026-01-24,card,shinhan,홈플러스㈜,single,68210,0,unknown,None,card_shinhan_25863165
1,2026-01-16,card,shinhan,홈플러스㈜,single,50240,0,unknown,None,card_shinhan_31299218
2,2026-01-16,card,shinhan,(주)아트박스 강서 홈플러스점,single,2000,0,unknown,None,card_shinhan_23635058
3,2026-01-07,card,shinhan,새서울약국,single,5700,0,unknown,None,card_shinhan_8159190
4,2026-01-07,card,shinhan,코아이비인후과,single,10800,0,unknown,None,card_shinhan_8112630


In [ ]:
expenditure_df.info()

## 3. 지출 분류

In [49]:
# 지출 매핑 파일을 불러옵니다.
from pathlib import Path

def find_project_root(
    start_path: Path | None = None,
    markers=("pyproject.toml", ".git")
) -> Path:
    """
    start_path부터 상위 디렉토리를 탐색하며
    markers 중 하나가 존재하는 디렉토리를 프로젝트 루트로 반환
    """
    if start_path is None:
        start_path = Path.cwd()

    current = start_path.resolve()

    for parent in [current] + list(current.parents):
        if any((parent / marker).exists() for marker in markers):
            return parent

    raise RuntimeError("프로젝트 루트를 찾을 수 없습니다.")

PROJECT_ROOT = find_project_root(Path.cwd())

In [50]:
mapping_path = PROJECT_ROOT / "data" / "resources" / "expenditure_mapping.csv"
# mapping_path.exists()

mapping_df = pd.read_csv(mapping_path)
mapping_df = mapping_df.sort_values("priority").reset_index(drop=True)
mapping_df.head()

,priority,match_type,pattern,category,comment
0,1,exact,지에스수퍼 김포한강자이점,식비,GS수퍼 고정 지점
1,1,contains,GS수퍼,식비,GS수퍼 일반
2,1,contains,지에스25,식비,GS25 편의점
3,1,contains,씨유,편의,CU 편의점
4,1,contains,홈플러스,식비,홈플러스 계열


In [51]:
import re

def map_category(merchant_name: str, mapping_df:pd.DataFrame) -> str | None:
    if pd.isna(merchant_name) or merchant_name == "":
        return None
    for _, row in mapping_df.iterrows():
        match_type = row["match_type"]
        pattern = row["pattern"]
        category = row["category"]
        if match_type == "exact":
            if merchant_name == pattern:
                return category
        elif match_type == "contains":
            if pattern in merchant_name:
                return category
        elif match_type == "regex":
            if re.search(pattern, merchant_name):
                return category
    return None


In [52]:
expenditure_df["category"] = expenditure_df["merchant_name"].apply(
    lambda x : map_category(x, mapping_df)
)
expenditure_df["category"] = expenditure_df["category"].fillna("미분류")

In [53]:
expenditure_df[["merchant_name", "category"]].head(20)

,merchant_name,category
0,홈플러스㈜,식비
1,홈플러스㈜,식비
2,(주)아트박스 강서 홈플러스점,식비
3,새서울약국,의료
4,코아이비인후과,기타
5,강서홈약국,의료
6,홈플러스㈜,식비


## 4. DB 저장

In [54]:
import sqlite3
db_path = PROJECT_ROOT / "data" / "db"/ "ledgerly.db"
conn = sqlite3.connect(db_path)

In [55]:
expenditure_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   used_at           7 non-null      object
 1   payment_type      7 non-null      object
 2   card_company      7 non-null      object
 3   merchant_name     7 non-null      string
 4   installment_type  7 non-null      object
 5   amount            7 non-null      int64 
 6   remaining_amount  7 non-null      int64 
 7   category          7 non-null      object
 8   memo              0 non-null      object
 9   source_uid        7 non-null      object
dtypes: int64(2), object(7), string(1)
memory usage: 692.0+ bytes


In [56]:
expenditure_df.to_sql(
    name="expenditure",
    con=conn,
    if_exists="append",
    index=False
)

7

In [57]:
fromdb_df =pd.read_sql("SELECT * FROM expenditure", conn)
fromdb_df

,id,used_at,payment_type,card_company,merchant_name,installment_type,amount,remaining_amount,category,memo,source_uid,created_at
0,1,2026-01-03,card,kb,쿠팡(쿠페이),single,32600,0,쇼핑,None,card_kb_30118172,2026-01-25 09:28:28
1,2,2026-01-03,card,kb,네이버페이,single,24840,0,쇼핑,None,card_kb_30118161,2026-01-25 09:28:28
2,3,2026-01-03,card,kb,네이버페이,single,47500,0,쇼핑,None,card_kb_30118150,2026-01-25 09:28:28
3,4,2026-01-03,card,kb,(주)딜리유통,single,42000,0,기타,None,card_kb_30118141,2026-01-25 09:28:28
4,5,2026-01-03,card,kb,웨이브(푹TV)자동,single,10900,0,문화,None,card_kb_30118138,2026-01-25 09:28:28
5,6,2026-01-01,card,kb,NICE 결제대행_4,single,110000,0,기타,None,card_kb_30118063,2026-01-25 09:28:28
6,7,2026-01-01,card,kb,구글플레이,single,19784,0,문화,None,card_kb_30118057,2026-01-25 09:28:28
7,8,2026-01-24,card,shinhan,홈플러스㈜,single,68210,0,식비,None,card_shinhan_25863165,2026-01-28 12:37:02
8,9,2026-01-16,card,shinhan,홈플러스㈜,single,50240,0,식비,None,card_shinhan_31299218,2026-01-28 12:37:02
9,10,2026-01-16,card,shinhan,(주)아트박스 강서 홈플러스점,single,2000,0,식비,None,card_shinhan_23635058,2026-01-28 12:37:02


In [58]:
fromdb_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id                14 non-null     int64 
 1   used_at           14 non-null     object
 2   payment_type      14 non-null     object
 3   card_company      14 non-null     object
 4   merchant_name     14 non-null     object
 5   installment_type  14 non-null     object
 6   amount            14 non-null     int64 
 7   remaining_amount  14 non-null     int64 
 8   category          14 non-null     object
 9   memo              0 non-null      object
 10  source_uid        14 non-null     object
 11  created_at        14 non-null     object
dtypes: int64(3), object(9)
memory usage: 1.4+ KB
